In [1]:
# === Imports ===

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from torch.utils.data import Dataset, DataLoader

import joblib
import os


In [3]:
# === Configuration ===

SEQ_LEN = 30
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-3

DATASET_PATH = "D:/ErgoSense/dataset/posture_dataset.csv"
MODEL_DIR = "D:/ErgoSense/models"

os.makedirs(MODEL_DIR, exist_ok=True)


In [5]:
# === Load dataset ===

df = pd.read_csv(DATASET_PATH)
df.head()


,nose_x,nose_y,nose_z,nose_visibility,left_shoulder_x,left_shoulder_y,left_shoulder_z,left_shoulder_visibility,right_shoulder_x,right_shoulder_y,...,left_ear_x,left_ear_y,left_ear_z,left_ear_visibility,right_ear_x,right_ear_y,right_ear_z,right_ear_visibility,timestamp,label
0,0.408185,0.568293,-2.250712,0.999495,0.794843,0.956978,-0.891110,0.989175,0.092592,0.934667,...,0.673248,3.897425,0.867931,0.000062,0.278294,3.883632,0.745527,1.145283e-05,1.763122e+09,1
1,0.393011,0.840976,-2.809807,0.996753,0.736628,0.897062,-1.970181,0.991840,0.099944,0.883424,...,0.660207,3.778626,1.759005,0.000192,0.282941,3.788914,2.374089,7.214556e-05,1.763197e+09,1
2,0.408060,0.501275,-1.238194,0.999920,0.727862,0.923024,-0.566044,0.997596,0.107714,0.926682,...,0.622256,3.287520,1.253096,0.000002,0.269836,3.273199,1.020084,2.195408e-07,1.763122e+09,1
3,0.436966,0.592251,-1.513547,0.999601,0.708134,0.920848,-0.526928,0.992406,0.173566,0.932093,...,0.612399,3.138843,0.833745,0.000059,0.311619,3.131513,0.703778,9.486115e-06,1.763198e+09,0
4,0.405768,0.865037,-2.464188,0.997987,0.763705,0.916441,-1.603421,0.995252,0.104482,0.895369,...,0.648273,3.744060,1.720935,0.000085,0.254696,3.759870,2.402974,2.598793e-05,1.763197e+09,1


In [7]:
# === Inspect data summary ===

print("Total samples:", len(df))
print("\nColumns:")
print(df.columns)

print("\nLabel distribution:")
print(df["label"].value_counts())


Total samples: 4000

Columns:
Index(['nose_x', 'nose_y', 'nose_z', 'nose_visibility', 'left_shoulder_x',
       'left_shoulder_y', 'left_shoulder_z', 'left_shoulder_visibility',
       'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z',
       'right_shoulder_visibility', 'left_elbow_x', 'left_elbow_y',
       'left_elbow_z', 'left_elbow_visibility', 'right_elbow_x',
       'right_elbow_y', 'right_elbow_z', 'right_elbow_visibility',
       'left_wrist_x', 'left_wrist_y', 'left_wrist_z', 'left_wrist_visibility',
       'right_wrist_x', 'right_wrist_y', 'right_wrist_z',
       'right_wrist_visibility', 'left_hip_x', 'left_hip_y', 'left_hip_z',
       'left_hip_visibility', 'right_hip_x', 'right_hip_y', 'right_hip_z',
       'right_hip_visibility', 'left_ear_x', 'left_ear_y', 'left_ear_z',
       'left_ear_visibility', 'right_ear_x', 'right_ear_y', 'right_ear_z',
       'right_ear_visibility', 'timestamp', 'label'],
      dtype='object')

Label distribution:
label
1    2000
0    2

In [9]:
# === Preprocessing ===

feature_cols = [col for col in df.columns if col not in ["label", "timestamp"]]
print("Using feature count:", len(feature_cols))

# Scale features
scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Save for inference
joblib.dump(scaler, f"{MODEL_DIR}/scaler.pkl")
print("Scaler saved!")


Using feature count: 44
Scaler saved!


In [11]:
# === Sequence Creation ===

def create_sequences(df, seq_len=30):
    X, y = [], []
    features = df[feature_cols].values
    labels = df["label"].values
    
    for i in range(len(df) - seq_len):
        X.append(features[i:i+seq_len])
        y.append(labels[i+seq_len-1])
        
    return np.array(X), np.array(y)

X, y = create_sequences(df, SEQ_LEN)

print("Sequence shape:", X.shape)
print("Labels shape:", y.shape)


Sequence shape: (3970, 30, 44)
Labels shape: (3970,)


In [13]:
# === Split ===

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Train: (3176, 30, 44)
Validation: (794, 30, 44)


In [15]:
# === PyTorch Dataset & Dataloader ===

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SequenceDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SequenceDataset(X_val, y_val), batch_size=BATCH_SIZE)


In [17]:
# === BiLSTM Model ===

class BiLSTMEncoder(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, num_classes=2):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        out, _ = self.bilstm(x)
        emb = out.mean(dim=1)
        logits = self.classifier(emb)
        return logits, emb


In [19]:
# === Training Loop ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMEncoder(input_size=X.shape[2]).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

best_acc = 0

for epoch in range(EPOCHS):

    # --- Training ---
    model.train()
    total_loss = 0
    
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(Xb)
        
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    # --- Validation ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits, _ = model(Xb)
            pred = logits.argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.3f} | Val Acc: {val_acc:.3f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), f"{MODEL_DIR}/bilstm_model.pth")
        print("Model saved!")

print("Best Accuracy:", best_acc)


Epoch 1/25 | Loss: 34.711 | Val Acc: 0.500
Model saved!
Epoch 2/25 | Loss: 34.702 | Val Acc: 0.501
Model saved!
Epoch 3/25 | Loss: 34.679 | Val Acc: 0.500
Epoch 4/25 | Loss: 34.677 | Val Acc: 0.500
Epoch 5/25 | Loss: 34.686 | Val Acc: 0.500
Epoch 6/25 | Loss: 34.678 | Val Acc: 0.500
Epoch 7/25 | Loss: 34.670 | Val Acc: 0.500
Epoch 8/25 | Loss: 34.655 | Val Acc: 0.500
Epoch 9/25 | Loss: 34.662 | Val Acc: 0.500
Epoch 10/25 | Loss: 34.667 | Val Acc: 0.509
Model saved!
Epoch 11/25 | Loss: 34.680 | Val Acc: 0.500
Epoch 12/25 | Loss: 34.661 | Val Acc: 0.500
Epoch 13/25 | Loss: 34.662 | Val Acc: 0.500
Epoch 14/25 | Loss: 34.661 | Val Acc: 0.500
Epoch 15/25 | Loss: 34.664 | Val Acc: 0.500
Epoch 16/25 | Loss: 34.664 | Val Acc: 0.500
Epoch 17/25 | Loss: 34.662 | Val Acc: 0.500
Epoch 18/25 | Loss: 34.657 | Val Acc: 0.500
Epoch 19/25 | Loss: 34.666 | Val Acc: 0.500
Epoch 20/25 | Loss: 34.672 | Val Acc: 0.500
Epoch 21/25 | Loss: 34.655 | Val Acc: 0.506
Epoch 22/25 | Loss: 34.684 | Val Acc: 0.500
Ep

In [21]:
# === Extract Embeddings ===

def extract_embeddings(loader):
    model.eval()
    embeddings, labels = [], []
    
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(device)
            _, emb = model(Xb)
            embeddings.append(emb.cpu().numpy())
            labels.append(yb.numpy())
    
    return np.vstack(embeddings), np.hstack(labels)

emb_train, y_train_svm = extract_embeddings(train_loader)
emb_val, y_val_svm = extract_embeddings(val_loader)

np.savez("D:/ErgoSense/dataset/bilstm_embeddings.npz",
         X_train=emb_train, y_train=y_train_svm,
         X_val=emb_val, y_val=y_val_svm)

print("Embeddings saved!")


Embeddings saved!
